# NF5 · Inteligencia de Negocio y toma de decisiones (RA5)
## El informe estratégico · *juego online*

**Tu misión:** convertir el Big Data del juego en **decisiones**. Construirás un
**Data Mart** agregado, lo visualizarás en un **dashboard** y escribirás un
**informe ejecutivo** para el estudio de videojuegos.

### Criterios del RA5
5.1/5.4 combinar dominios · 5.2 limpieza/transformación · 5.3 Big Data potencia el BI
· 5.5 interpretar y decidir · 5.6 simular la implantación BI.

> Autoevaluable en su parte de **Data Mart** (KPIs). El dashboard y el informe se
> evalúan con revisión asistida. Herramienta **a tu elección**: Power BI (Windows)
> o **Tableau Public** (Mac/Windows, gratis).

### Antes de empezar · genera los datos (una sola vez)

Abre una **terminal** (no una celda) y ejecuta, **desde la raíz del repositorio**:

```bash
cd nf5
python datos/generar_datos.py --salida datos/raw
```

Este cuaderno **se sitúa solo** en `nf5/`, así que todas sus rutas son relativas a esa
carpeta. Si la primera celda de código falla con un error de fichero no encontrado, es
que te falta este paso.


In [ ]:
import os
if os.path.basename(os.getcwd()) == "actividad": os.chdir("..")  # ejecutar desde la carpeta del núcleo (donde está datos/)
import glob
# --- Fuerza Java 17 para Spark (Java 18+ provoca el error getSubject) ---
_j = sorted(glob.glob("/usr/lib/jvm/*17*") + glob.glob("/usr/local/sdkman/candidates/java/17*"))
if _j:
    os.environ["JAVA_HOME"] = _j[0]; os.environ["PATH"] = _j[0] + "/bin:" + os.environ.get("PATH", "")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
print("JAVA_HOME =", os.environ.get("JAVA_HOME", "(no encontrado: reconstruye el Codespace)"))
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

import json, shutil
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.appName("NF5").master("local[*]")
         .config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
RAW, MART = Path("datos/raw"), Path("datamart"); MART.mkdir(exist_ok=True)
resultados = {}
print("Spark listo.")

---
## Fase 1 · Selección de datos (RA5.1/5.4)
Carga los **tres dominios**: `eventos.parquet` (operacional), `alertas.parquet`
(incidentes) y `regiones.csv` (la dimensión del mapa).

In [ ]:
# Fase 1 · Selección de los 3 dominios
eventos = ...   # TODO: lee eventos.parquet (partidas) · pista: spark.read.parquet(f"{RAW}/eventos.parquet")
alertas = ...   # TODO: lee alertas.parquet (reportes) · pista: spark.read.parquet(...)
regiones = ...     # TODO: lee regiones.csv (regiones) con cabecera e inferSchema · pista: .option("header",True).option("inferSchema",True).csv(...)
resultados["fase1_seleccion"] = {"filas_eventos": int(eventos.count()),
                                 "filas_alertas": int(alertas.count()), "n_dominios": 3}
resultados["fase1_seleccion"]

---
## Fase 2 · ETL → Data Mart agregado (RA5.2/5.3/5.4)
Combina los dominios y **agrega** a un Data Mart pequeño (lo que consumirá el BI):
1. `fact_región.csv`: por región → `n_eventos`, `n_alertas`, `n_criticas`.
2. `fact_tipo.csv`: valor medio por tipo.
3. `dim_region.csv`: la dimensión de zonas del mapa.

> **Clave (examen):** el dashboard se conecta a este Data Mart agregado, NO al Big
> Data crudo → dashboards rápidos y buena experiencia de usuario.

In [ ]:
# Fase 2 · ETL -> Data Mart agregado
ev_d = ...   # TODO: une eventos con regiones por id_zona (trae "region") · pista: eventos.join(regiones.select("id_zona","region"), "id_zona")
al_d = ...   # TODO: une alertas con regiones por id_zona (trae "region")

eventos_region = ev_d.groupBy("region").count().withColumnRenamed("count","n_eventos")
alertas_region = (al_d.groupBy("region")
    .agg(F.count("*").alias("n_alertas"),
         F.sum(F.when(F.col("severidad")=="critica",1).otherwise(0)).alias("n_criticas")))
fact_region = eventos_region.join(alertas_region,"region","outer").fillna(0).orderBy("region")
valor_tipo = eventos.groupBy("tipo").agg(F.round(F.avg("valor"),2).alias("valor_medio")).orderBy("tipo")

fact_region.toPandas().to_csv(MART/"fact_region.csv", index=False)
valor_tipo.toPandas().to_csv(MART/"fact_tipo.csv", index=False)
regiones.toPandas().to_csv(MART/"dim_region.csv", index=False)
print("Data Mart escrito en", MART)

---
## Fase 3 · KPIs (RA5.3/5.5)
Calcula los KPIs de decisión a partir del Data Mart.

In [ ]:
# Fase 3 · KPIs (a partir del Data Mart)
fd = {int(r["region"]): r for r in fact_region.collect()}
eventos_por_region = {str(d): int(fd[d]["n_eventos"]) for d in sorted(fd)}
criticas_por_region = {str(d): int(fd[d]["n_criticas"]) for d in sorted(fd)}
valor_medio_por_tipo = {r["tipo"]: float(r["valor_medio"]) for r in valor_tipo.collect()}
total_criticas = sum(criticas_por_region.values()); total_alertas = alertas.count()
top_region = ...   # TODO: la región con MÁS reportes críticos · pista: max(dic, key=dic.get) sobre criticas_por_region

resultados["datamart"] = {"n_tablas": 3, "eventos_por_region": eventos_por_region,
    "criticas_por_region": criticas_por_region,
    "valor_medio_por_tipo": {k: valor_medio_por_tipo[k] for k in sorted(valor_medio_por_tipo)}}
resultados["kpi"] = {"tasa_critica_global": round(total_criticas/total_alertas,4),
    "top_region_critico": int(top_region), "total_eventos": int(eventos.count())}
print("Top region por alertas críticas:", top_region)
resultados["kpi"]

---
## Fase 4 · Dashboard (RA5.6) — Power BI **o** Tableau Public
Conecta tu herramienta a los CSV de `datamart/` y construye un dashboard para un
equipo del estudio (al menos: eventos por región, alertas críticas por región,
valor medio por tipo, y un KPI destacado).
- **Tableau Public** (gratis, Mac/Win): *Connect → Text file* → arrastra los CSV.
- **Power BI Desktop** (Windows): *Obtener datos → Texto/CSV*.

Exporta el dashboard (`.twbx` o `.pbix`, o un PDF) a la carpeta `entrega/` y
declara la herramienta abajo.

In [ ]:
# Declara tu entrega del dashboard
HERRAMIENTA = "tableau"        # "tableau" o "powerbi"
FICHERO_ENTREGADO = True       # ponlo a True cuando hayas exportado el dashboard a entrega/
resultados["dashboard"] = {"herramienta": HERRAMIENTA, "fichero_entregado": FICHERO_ENTREGADO}

---
## Fase 5 · Informe ejecutivo (RA5.5) — texto

Escribe un informe breve (10-15 líneas) para el equipo del estudio con:

- La **interpretación** del hallazgo principal (mira `top_region_critico` en tus KPIs).
- **2 decisiones accionables**: ¿dónde reasignas recursos y por qué?
- La distinción entre una **observación** y un **insight accionable**:
  - *Observación:* «la región 3 concentra el 56 % de los reportes críticos».
  - *Insight accionable:* «asignar 2 moderadores más al turno de la región 3 bajaría el
    tiempo de respuesta a los reportes un 40 %, con el coste de dejar la región 1 con uno».

La diferencia no es el tono: un insight dice **qué hace alguien mañana**, **quién** y **a
cambio de qué**.

*(Escribe aquí tu informe.)*


---
## Fase 6 · Razonamiento (formato examen, RA5)
En **máximo 15 líneas**:

**a)** Justifica la importancia arquitectónica de crear un **Data Mart agregado**
antes de conectar Power BI/Tableau (rendimiento del dashboard y experiencia del
usuario final).

**b)** Distingue **observación** de **insight accionable** y aplícalo a una decisión
estratégica de retención/monetización.

*(Escribe aquí tu respuesta.)*


### (Opcional, sin coste) · GenAI / text-to-SQL
Si quieres, sube `datamart/` a **Databricks Free Edition** y usa **Genie** para
preguntar en lenguaje natural (p.ej. *"¿qué región tiene más alertas críticas?"*).
Es opcional, gratuito y no se evalúa: solo para que veas el text-to-SQL en acción.

---
## Celda final · Generar `resultados.json` (no modificar)

In [ ]:
# Pega aquí tus 2 decisiones accionables de la Fase 5 (el informe completo va arriba,
# en markdown; esto es solo para que la corrección compruebe que existen).
DECISION_PROPUESTA = """
...
"""

resultados["insight"] = {"decision_propuesta_texto_len": len(DECISION_PROPUESTA.strip())}
if resultados["insight"]["decision_propuesta_texto_len"] < 120:
    print("⚠️  Autochequeo: DECISION_PROPUESTA es muy corto o está sin rellenar.")

ALUMNO = "TU_NOMBRE_Y_APELLIDOS"   # <-- pon aquí "Apellidos, Nombre"
resultados["metadata"] = {"caso":"gaming","seed":42,"alumno":ALUMNO}
assert ALUMNO != "TU_NOMBRE_Y_APELLIDOS", "⚠️ Pon tus Apellidos, Nombre en ALUMNO antes de entregar."
faltan = {"fase1_seleccion","datamart","kpi","dashboard"} - set(resultados)
assert not faltan, f"⚠️ Faltan secciones: {sorted(faltan)}. Ejecuta TODAS las fases (incluida la Fase 4 · dashboard) antes de generar la entrega."
json.dump(resultados, open("resultados.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
spark.stop()
print("✅ resultados.json generado. Entrega el .ipynb, datamart/, el dashboard exportado y el informe.")

---
### Checkpoint de preparación al examen (no evaluable)
1. ¿Por qué conectar Power BI al Data Lake crudo da dashboards lentos?
2. "El región 3 concentra el 56% de las críticas": ¿es observación o insight?
3. ¿Qué aporta agregar los datos ANTES de visualizar frente a hacerlo en la herramienta BI?